<a href="https://colab.research.google.com/github/Dhanavijayan/ML_OPS_Tasks/blob/main/yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.2/112.6 GB disk)


In [9]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [10]:
!ln -s /content/gdrive/My\ Drive/ /mydrive
!ls /mydrive

 1_page.pdf
'2027 IT Training Time Table and Batch Details.xlsx'
'2027_Rocket India(Mr.Cooper)_SIET DB ver1.0.xlsx'
'714023243031_DHANAVIJAYAN_M (1).pdf'
 714023243031_DHANAVIJAYAN_M.pdf
'AI&DS 2027 IT Placement Training Details-1.xlsx'
'Batch New.xlsx'
'Colab Notebooks'
'Copy of Dataset .csv'
'Copy of Machine Learning Internship Task .pdf'
 EAadhaar_2726111142689220220805151452_04042024212144.pdf
 extracted_frames
'IMG-20220408-WA0019 (1).jpg'
 IMG-20220408-WA0019.jpg
 IMG-20220408-WA0020.jpg
 IMG-20231003-WA0007.jpg
 IMG-20231003-WA0010.jpg
 intern.zip
 label
'python basics certificate.pdf'
'Saved from Chrome'
'Screenshot 2026-02-17 at 10.30.33 AM.png'
'Screenshot 2026-02-17 at 10.31.34 AM.png'
 SREC_Bus_Route.pdf
'Students Performance Report - 2027 Batch - v1 (1).xlsx'
'Supermarket Theft Identification System.pdf'
'WhatsApp Image 2025-02-26 at 20.46.42 (1).jpeg'


In [11]:
import zipfile
import os
import shutil

gdrive_root = '/content/gdrive/My Drive/'
zip_file_path = os.path.join(gdrive_root, 'Accident.zip')
labels_source_path = os.path.join(gdrive_root, 'label')

unzipped_base_dir = '/content/Accident_unzipped/'
unzipped_images_actual_dir = os.path.join(unzipped_base_dir, 'Accident')
dataset_root = '/content/yolov8_dataset/'
train_images_dir = os.path.join(dataset_root, 'images', 'train')
train_labels_dir = os.path.join(dataset_root, 'labels', 'train')

os.makedirs(unzipped_base_dir, exist_ok=True)
os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)

if os.path.exists(zip_file_path):
    print(f"Unzipping '{zip_file_path}' to '{unzipped_base_dir}'...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(unzipped_base_dir)
    print("Unzipping complete.")
else:
    print(f"Error: Zip file not found at '{zip_file_path}'. Please check the path and ensure it's in your Drive root.")

print(f"Copying images from '{unzipped_images_actual_dir}' to '{train_images_dir}'...")
image_files_copied = 0
if os.path.exists(unzipped_images_actual_dir):
    for filename in os.listdir(unzipped_images_actual_dir):
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy(os.path.join(unzipped_images_actual_dir, filename), train_images_dir)
            image_files_copied += 1
else:
    print(f"Error: Image source directory '{unzipped_images_actual_dir}' not found after unzipping. Check the zip file structure.")
print(f"Copied {image_files_copied} image files.")

if os.path.exists(labels_source_path):
    print(f"Copying label files from '{labels_source_path}' to '{train_labels_dir}'...")
    label_files_copied = 0
    for filename in os.listdir(labels_source_path):
        if filename.endswith('.txt'):
            shutil.copy(os.path.join(labels_source_path, filename), train_labels_dir)
            label_files_copied += 1
    print(f"Copied {label_files_copied} label files.")
else:
    print(f"Error: Label directory not found at '{labels_source_path}'. Please check the path and ensure it's in your Drive root.")

print(f"Dataset prepared at '{dataset_root}'.")

Error: Zip file not found at '/content/gdrive/My Drive/Accident.zip'. Please check the path and ensure it's in your Drive root.
Copying images from '/content/Accident_unzipped/Accident' to '/content/yolov8_dataset/images/train'...
Copied 362 image files.
Copying label files from '/content/gdrive/My Drive/label' to '/content/yolov8_dataset/labels/train'...
Copied 362 label files.
Dataset prepared at '/content/yolov8_dataset/'.


In [12]:
data_yaml_content = f"""
path: {dataset_root} # Dataset root directory
train: images/train    # Train images relative to 'path'
val: images/train      # Val images relative to 'path' (using train for simplicity, as no separate val set was mentioned)
nc: 1                  # Number of classes
names: ['accident']    # Class names
"""

data_yaml_path = os.path.join(dataset_root, 'data.yaml')

with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"'data.yaml' created at '{data_yaml_path}':")
with open(data_yaml_path, 'r') as f:
    print(f.read())


'data.yaml' created at '/content/yolov8_dataset/data.yaml':

path: /content/yolov8_dataset/ # Dataset root directory
train: images/train    # Train images relative to 'path'
val: images/train      # Val images relative to 'path' (using train for simplicity, as no separate val set was mentioned)
nc: 1                  # Number of classes
names: ['accident']    # Class names



In [17]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(data=os.path.join(dataset_root, 'data.yaml'), epochs=50, imgsz=640, batch=16, device=0, project='runs/detect', name='train_accident_detection_yolov8n')

print("Training process initiated. Check the output above for progress and results. The trained model and logs will be saved in '/content/runs/detect/train_accident_detection_yolov8n/'.")

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolov8_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_accident_detection_yolov8n-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

In [18]:
from ultralytics import YOLO
import os


runs_dir = '/content/runs/detect/runs/detect/'
latest_run = None
if os.path.exists(runs_dir):
    run_folders = [f for f in os.listdir(runs_dir) if os.path.isdir(os.path.join(runs_dir, f))]
    if run_folders:

        run_folders.sort(key=lambda x: os.path.getmtime(os.path.join(runs_dir, x)), reverse=True)
        latest_run = run_folders[0]

if latest_run:
    model_path = os.path.join(runs_dir, latest_run, 'weights', 'best.pt')
    print(f"Loading model from: {model_path}")
else:
    print("Warning: Could not automatically find the latest run directory. Using a default path.")
    model_path = '/content/runs/detect/train_accident_detection_yolov8n/weights/best.pt' # Fallback default



model = YOLO(model_path)


metrics = model.val(data=os.path.join(dataset_root, 'data.yaml'))


print("\nModel Evaluation Metrics:")
print(f"  mAP50: {metrics.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP50-95: {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")



Loading model from: /content/runs/detect/runs/detect/train_accident_detection_yolov8n-2/weights/best.pt
Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2500.5±452.1 MB/s, size: 206.9 KB)
val: Scanning /content/yolov8_dataset/labels/train.cache... 362 images, 0 backgrounds, 27 corrupt: 100% ━━━━━━━━━━━━ 362/362 89.3Mit/s 0.0s
train: /content/yolov8_dataset/images/train/test10_10.jpg: ignoring corrupt image/label: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (64,) + inhomogeneous part.
train: /content/yolov8_dataset/images/train/test10_11.jpg: ignoring corrupt image/label: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (64,) + inhomogeneous part.
train: /co

In [19]:
import shutil

if 'latest_run' in globals() and latest_run:
    source_model_path = os.path.join(runs_dir, latest_run, 'weights', 'best.pt')
else:
    print("Error: Could not determine the latest run directory. Please manually specify 'source_model_path'.")
    source_model_path = '/content/runs/detect/train_accident_detection_yolov8n-2/weights/best.pt' # Fallback default

gdrive_save_dir = os.path.join(gdrive_root, 'trained_models')
os.makedirs(gdrive_save_dir, exist_ok=True)

destination_model_path = os.path.join(gdrive_save_dir, 'yolov8n_accident_detection_best.pt')

print(f"Attempting to save model from: {source_model_path}")
print(f"To Google Drive at: {destination_model_path}")

try:
    if os.path.exists(source_model_path):
        shutil.copy(source_model_path, destination_model_path)
        print(f"Successfully saved the trained model to Google Drive at: {destination_model_path}")
    else:
        print(f"Error: Source model file not found at {source_model_path}")
except Exception as e:
    print(f"An error occurred while saving the model: {e}")


Attempting to save model from: /content/runs/detect/runs/detect/train_accident_detection_yolov8n-2/weights/best.pt
To Google Drive at: /content/gdrive/My Drive/trained_models/yolov8n_accident_detection_best.pt
Successfully saved the trained model to Google Drive at: /content/gdrive/My Drive/trained_models/yolov8n_accident_detection_best.pt
